In [7]:
import pandas as pd
import feedparser
from src.collectors.rss_utils import collect_rss_articles, clean_url, google_news_resolve_many
from src.collectors.ccnews_extractor import parse_extracted_article
from src.config import get_settings

import json
from urllib.parse import urlparse, urlsplit, urlunsplit, parse_qsl, urlencode
from pathlib import Path
import time
import os
from datetime import datetime
import random
import requests
import trafilatura
from tqdm import tqdm
import polars as pl
from hashlib import sha256
from src.pipeline.load_ingest_to_db import insert_polars_to_postgres, scan_tree

## check rss links

In [2]:
ALLOWED_DOMAINS = get_settings().allowed_domains
domains = sorted(ALLOWED_DOMAINS)

manual_feeds = {
    "bbc.com": ["https://feeds.bbci.co.uk/news/world/rss.xml"],
    "bbc.co.uk": ["https://feeds.bbci.co.uk/news/world/rss.xml"],
    "nytimes.com": ["https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml"],
    "cnn.com": ["https://rss.cnn.com/rss/edition.rss"],
    "aljazeera.com": ["https://www.aljazeera.com/xml/rss/all.xml"],
    "abcnews.go.com": ["https://abcnews.go.com/abcnews/topstories"],
}

paths = [
    "/feed",
    "/rss",
    "/rss.xml",
    "/feed.xml",
    "/atom.xml",
    "/index.xml",
    "/news/rss",
    "/rss/news",
]

def candidate_urls(domain: str) -> list[str]:
    base = [f"https://{domain}", f"http://{domain}"]
    if domain.startswith("www."):
        bare = domain[4:]
        base += [f"https://{bare}", f"http://{bare}"]
    urls = []
    for b in base:
        urls.extend([b + p for p in paths])
    urls.extend(manual_feeds.get(domain, []))
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            out.append(u)
            seen.add(u)
    return out


In [6]:

rows = []
working_feeds = []


print(f"Starting RSS discovery for {len(domains)} domains...\n")
for i, d in enumerate(domains, 1):
    found = None
    found_count = 0
    found_title = ""

    urls = candidate_urls(d)
    print(f"[{i}/{len(domains)}] domain={d} candidates={len(urls)}")

    for j, u in enumerate(urls, 1):
        f = feedparser.parse(u)
        bozo_ok = int(getattr(f, "bozo", 0)) == 0
        n = len(getattr(f, "entries", []))
        title = str(getattr(f, "feed", {}).get("title", "")).strip()

        print(
            f"   - try {j:02d}: entries={n:4d} bozo_ok={bozo_ok} "
            f"url={u}"
        )

        if n > 0:
            found = u
            found_count = n
            found_title = title
            break
        if bozo_ok and title:
            found = u
            found_count = n
            found_title = title
            break
    rows.append(
        {
            "domain": d,
            "rss_found": bool(found),
            "feed_url": found,
            "entries": found_count,
            "feed_title": found_title,
        }
    )
    if found:
        working_feeds.append(found)

rss_check_df = pd.DataFrame(rows).sort_values(["rss_found", "entries"], ascending=[False, False]).reset_index(drop=True)
print("domains_total:", len(domains))
print("domains_with_rss:", int(rss_check_df["rss_found"].sum()))
print("working_feeds:", len(working_feeds))

articles = collect_rss_articles(working_feeds)
articles_df = pd.DataFrame([a.__dict__ for a in articles]).drop_duplicates(subset=["url"]).reset_index(drop=True)

print("parsed_articles_unique_urls:", len(articles_df))
display(rss_check_df)
articles_df.head(20)

Starting RSS discovery for 53 domains...

[1/53] domain=abcnews.go.com candidates=17
   - try 01: entries=   0 bozo_ok=True url=https://abcnews.go.com/feed
   - try 02: entries=   0 bozo_ok=True url=https://abcnews.go.com/rss
   - try 03: entries=   0 bozo_ok=False url=https://abcnews.go.com/rss.xml
   - try 04: entries=   0 bozo_ok=True url=https://abcnews.go.com/feed.xml
   - try 05: entries=   0 bozo_ok=True url=https://abcnews.go.com/atom.xml
   - try 06: entries=   0 bozo_ok=True url=https://abcnews.go.com/index.xml
   - try 07: entries=   0 bozo_ok=True url=https://abcnews.go.com/news/rss
   - try 08: entries=   0 bozo_ok=True url=https://abcnews.go.com/rss/news
   - try 09: entries=   0 bozo_ok=True url=http://abcnews.go.com/feed
   - try 10: entries=   0 bozo_ok=True url=http://abcnews.go.com/rss
   - try 11: entries=   0 bozo_ok=True url=http://abcnews.go.com/rss.xml
   - try 12: entries=   0 bozo_ok=True url=http://abcnews.go.com/feed.xml
   - try 13: entries=   0 bozo_ok=Tru

RemoteDisconnected: Remote end closed connection without response

## parse rss

In [7]:
# RSS_URLs = ["https://www.aljazeera.com/xml/rss/all.xml"]  # same feed as /rss self-link
RSS_URLs = get_settings().rss_feeds
MAX_ITEMS = 200
MIN_TEXT_LEN = 500
REQUEST_TIMEOUT = 30
ALLOWED_LANGS = {"en", "ru"}  # keep same policy as CC pipeline

root_dir = Path(os.path.abspath(os.path.join(os.path.dirname('__file__')))).parent
OUT_ROOT = os.path.join(root_dir, "data", "raw", "rss")

feed = feedparser.parse(RSS_URLs[0])
entries = feed.entries[:MAX_ITEMS]


In [ ]:

items = []
stats = {
    "rss_entries": len(entries),
    "fetched_ok": 0,
    "extract_non_empty": 0,
    "json_errors": 0,
    "lang_filtered_out": 0,
    "written": 0,
}

urls = [clean_url(str(e.get("link", "")).strip()) for e in entries]
resolved_urls_dict = await google_news_resolve_many(urls)
resolved_urls = list(resolved_urls_dict.values())

for url in tqdm(resolved_urls, desc="RSS -> article extraction"):

    if not url:
        continue

    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT, headers={"User-Agent": "Mozilla/5.0"})        
        r.raise_for_status()
        html = r.text
        stats["fetched_ok"] += 1
    except Exception:
        continue

    extracted = trafilatura.extract(
        html,
        output_format="json",
        with_metadata=True
    )

    parsed, d = parse_extracted_article(
        extracted,
        min_text_len=MIN_TEXT_LEN,
        allowed_langs=ALLOWED_LANGS,
        lang_sample_chars=500,
        )
        
    for k, v in d.items():
        stats[k] = stats.get(k, 0) + v
    if not parsed:
        continue

    data, text, lang = parsed["data"], parsed["text"], parsed["lang"]

    item = {
        "url": url,
        "domain": urlparse(url).netloc.lower(),
        "title": data.get("title") or e.get("title"),
        "date": data.get("date") or e.get("published"),
        "author": data.get("author"),
        "lang": lang,
        "text": text,
    }
    items.append(item)
    stats["written"] += 1

print(stats)
print("sample:", items[0] if items else None)

RSS -> article extraction: 100%|██████████| 25/25 [00:30<00:00,  1.23s/it]

{'rss_entries': 25, 'fetched_ok': 25, 'extract_non_empty': 25, 'json_errors': 0, 'lang_filtered_out': 0, 'written': 25, 'lang_errors': 0}
sample: {'url': 'https://www.bbc.com/news/articles/cy91r7ww3weo?at_medium=RSS&at_campaign=rss', 'domain': 'www.bbc.com', 'title': "Australia's most-decorated soldier vows to fight war crime charges", 'date': '2026-04-19', 'author': 'Tabby Wilson', 'lang': 'en', 'text': 'Australia\'s most-decorated soldier vows to fight war crime charges\nAustralia\'s most decorated living soldier, Ben Roberts-Smith, has publicly denied all allegations against him in his first statement after being charged with five counts of the war crime of murder last week.\nThe Victoria Cross recipient, released on bail on Friday, said he was "proud of my service in Afghanistan", and would use the charges against him as an opportunity to "finally" clear his name.\nHe said: "I understand this journey will be difficult. But I can promise everybody that I have never run from a fight 

In [12]:
print(json.dumps(items[0], indent=4), end='\\n')

{
    "url": "https://www.bbc.com/news/articles/c78rr9r4e5po",
    "domain": "www.bbc.com",
    "title": "Another woman accuses Eric Swalwell of drugging and raping her in 2018",
    "date": "2026-04-14",
    "author": "Madeline Halpert",
    "lang": "en",
    "text": "Another woman accuses Swalwell of rape, saying he drugged her in 2018\nAnother woman has alleged that Eric Swalwell raped her, adding to a growing list of misconduct allegations against the former Democratic lawmaker.\nSpeaking on Tuesday, Lonna Drewes said she had been reluctant to come forward with her allegations that Swalwell drugged and raped her in a hotel room in 2018 because of his \"political power\".\nSwalwell has resigned from Congress and withdrawn from California governor's race since accusations against him were reported on Friday. He denies \"each and every\" sexual misconduct allegation against him.\n\"These accusations are false, fabricated, and deeply offensive - a calculated and transparent political h

In [ ]:

def _month_partition_path(base_dir: str, item_date: str | None) -> str:
    dt = pd.to_datetime(item_date, errors="coerce")
    if pd.isna(dt):
        dt = pd.Timestamp.utcnow()
    year = f"{dt.year:04d}"
    month = f"{dt.month:02d}"
    out_dir = os.path.join(base_dir, year, month)
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, "articles.jsonl")

def run_rss_cycle(
    rss_urls,
    *,
    max_items=200,
    min_text_len=500,
    allowed_langs=("en", "ru"),
    request_timeout=30,
    out_root=OUT_ROOT
):
    items = []
    seen_urls = set()
    stats = {
        "feeds": len(rss_urls),
        "rss_entries": 0,
        "fetched_ok": 0,
        "extract_non_empty": 0,
        "json_errors": 0,
        "lang_errors": 0,
        "lang_filtered_out": 0,
        "written": 0,
    }

    for rss_url in rss_urls:
        feed = feedparser.parse(rss_url)
        entries = feed.entries[:max_items]
        stats["rss_entries"] += len(entries)

        for e in tqdm(entries, desc=f"RSS -> extract | {rss_url}"):
            
            time.sleep(random.uniform(0.5, 1.2))
            url = clean_url(str(e.get("link", "")).strip())
            if not url or url in seen_urls:
                continue
            seen_urls.add(url)

            try:
                r = requests.get(
                    url,
                    timeout=request_timeout,
                    headers={"User-Agent": "Mozilla/5.0"},
                )
                r.raise_for_status()
                html = r.text
                stats["fetched_ok"] += 1
            except Exception:
                continue

            extracted = trafilatura.extract(
                html,
                output_format="json",
                with_metadata=True
            )

            parsed, d = parse_extracted_article(
                extracted,
                min_text_len=min_text_len,
                allowed_langs=allowed_langs,
                lang_sample_chars=500,
            )
            for k, v in d.items():
                stats[k] = stats.get(k, 0) + v
            if not parsed:
                continue

            data, text, lang = parsed["data"], parsed["text"], parsed["lang"]
            item = {
                "url": url,
                "domain": urlparse(url).netloc.lower(),
                "title": data.get("title") or e.get("title"),
                "date": data.get("date") or e.get("published"),
                "author": data.get("author"),
                "lang": lang,
                "text": text,
                "rss_url": rss_url,
                "collected_at": datetime.utcnow().isoformat(),
            }
            items.append(item)

        # save partitioned by item month
        for x in items:
            out_path = _month_partition_path(out_root, x.get("date"))
            with open(out_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(x, ensure_ascii=False) + "\n")
            stats["written"] += 1

    return stats, items

In [ ]:
t0 = time.time()
stats, _ = run_rss_cycle(RSS_URLs, out_root=OUT_ROOT)
elapsed = time.time() - t0
print(f"[RSS cycle done] elapsed_s={elapsed:.2f} stats={stats}")

RSS -> extract | https://feeds.bbci.co.uk/news/world/rss.xml: 100%|██████████| 26/26 [00:49<00:00,  1.89s/it]
RSS -> extract | https://feeds.bbci.co.uk/news/business/rss.xml: 100%|██████████| 49/49 [01:49<00:00,  2.23s/it]
RSS -> extract | https://feeds.bbci.co.uk/news/politics/rss.xml: 100%|██████████| 61/61 [02:07<00:00,  2.09s/it]
RSS -> extract | https://feeds.bbci.co.uk/news/technology/rss.xml: 100%|██████████| 21/21 [00:35<00:00,  1.71s/it]


[RSS cycle done] elapsed_s=332.13 stats={'feeds': 4, 'rss_entries': 157, 'fetched_ok': 143, 'extract_non_empty': 143, 'json_errors': 0, 'lang_errors': 0, 'lang_filtered_out': 0, 'written': 141}


RSS -> extract | https://feeds.bbci.co.uk/news/world/rss.xml:  73%|███████▎  | 19/26 [00:31<00:11,  1.67s/it]


KeyboardInterrupt: 

In [ ]:
settings = get_settings()
rss_root = Path("/opt/airflow/project/data/raw/rss")
jsonl_files = [p for p in rss_root.rglob("*.jsonl") if p.is_file()]
if not jsonl_files:
    print({"rows_in_df": 0, "inserted_estimate": 0})

# rss_lf = scan_tree(rss_root).select(
rss_lf = scan_tree(rss_root)
cols = set(rss_lf.collect_schema().names())
datetime_col = (
        pl.col("datetime").cast(pl.Utf8)
        if "datetime" in cols
        else pl.lit(None, dtype=pl.Utf8)
)
date_col = (
    pl.col("date").cast(pl.Utf8)
    if "date" in cols
    else pl.lit(None, dtype=pl.Utf8)
)

rss_lf = rss_lf.select([
    # [
        pl.lit("rss").alias("source_type"),
        pl.col("domain").cast(pl.Utf8).str.to_lowercase().alias("domain"),
        pl.col("title").cast(pl.Utf8).alias("title"),
        pl.col("url").cast(pl.Utf8).alias("url"),
        # pl.coalesce([
        #     pl.col("datetime").cast(pl.Utf8).str.strptime(pl.Datetime, strict=False),
        #     pl.col("date").cast(pl.Utf8).str.strptime(pl.Datetime, strict=False),
        # ]).alias("datetime"),
        pl.coalesce([datetime_col, date_col + pl.lit("T00:00:00")]).alias("datetime_str"),
        date_col.alias("date_str"),
        # pl.col("date").cast(pl.Utf8).str.strptime(pl.Date, strict=False).alias("date"),
        pl.col("author").cast(pl.Utf8).alias("author"),
        pl.col("lang").cast(pl.Utf8).alias("lang"),
        pl.col("text").cast(pl.Utf8).alias("text"),
    ]
).with_columns([
    pl.col("datetime_str").str.to_datetime(strict=False).alias("datetime"),
    pl.col("date_str").str.strptime(pl.Date, strict=False).alias("date"),
]
).filter(
    pl.col("url").is_not_null()
    & pl.col("text").is_not_null()
    & (pl.col("url").str.len_chars() > 0)
    & (pl.col("text").str.len_chars() > 0)
)

rss_lf = rss_lf.with_columns(
    [
        pl.col("url")
        .map_elements(
            lambda u: sha256(u.encode("utf-8")).hexdigest(),
            return_dtype=pl.Utf8,
        )
        .alias("article_id"),
        pl.col("text")
        .map_elements(
            lambda t: sha256(t[:500].lower().strip().encode("utf-8")).hexdigest(),
            return_dtype=pl.Utf8,
        )
        .alias("text_hash"),
    ]
)

df = (
    rss_lf.select(
        [
            "article_id",
            "text_hash",
            "source_type",
            "domain",
            "title",
            "url",
            "datetime",
            "date",
            "author",
            "lang",
            "text",
        ]
    )
    .unique(subset=["article_id"], keep="first")
    .collect(streaming=True)
)

stats = insert_polars_to_postgres(
    df,
    table_name="news_articles",
    target_cols=[
        "article_id",
        "text_hash",
        "source_type",
        "domain",
        "title",
        "url",
        "datetime",
        "date",
        "author",
        "lang",
        "text",
    ],
    dsn=settings.postgres_dsn,
    conflict_col="article_id",
    batch_size=5000,
    verbose=True,
)
return stats